# Collaborative Filtering using Singular Value Decomposition (SVD), Matrix Factorization

This notebook implements a CF recommendation system using Singular Value Decomposition (SVD) within our Matrix Factorization. The pipeline follows these steps: load user-item interaction data, construct a "sparse" user-item matrix, impute the missing values using a cascaded mean approach -- which means user mean, fallback to item mean, fallback to global mean, apply SVD like we have in class to capture latent user-item features, and finally evaluate.

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse.linalg import svds
from shared import load_data, evaluate_model, evaluate_rating_metrics

train_df, test_df, restaurants_df = load_data()

## User-Item Matrix Construction

Here, we create the unique identifier mappings for both users and items. Essentially unique users are assigned row numbers and unique restaurants are assigned column numbers. This is necessary to construct our 2D numpy array! The values within the array correspond to the star ratings users have given restaurants. Items that a user hasn't rated yet are initialized with a value of `0`.

In [ ]:
user_ids = train_df['user_id'].unique()
item_ids = restaurants_df['business_id'].unique()

user_to_idx = {user: idx for idx, user in enumerate(user_ids)}
idx_to_user = {idx: user for user, idx in user_to_idx.items()}

item_to_idx = {item: idx for idx, item in enumerate(item_ids)}
idx_to_item = {idx: item for item, idx in item_to_idx.items()}

num_users = len(user_ids)
num_items = len(item_ids)
R = np.zeros((num_users, num_items))

for row in train_df.itertuples():
    u_idx = user_to_idx[row.user_id]
    i_idx = item_to_idx[row.business_id]
    R[u_idx, i_idx] = row.stars

## Missing Value Imputation

Since traditional SVD (what we're using) requires a matrix to be fully populated, we need to impute the values that are missing (currently being represented as 0's). To do this we use a "cascading" strategy where we have fallbacks in case the others don't work:

1. **User Mean**: Fills missing values with the specific user's average rating (this captures how lenient or strict the user is).
2. **Item Mean**: If a user has no ratings available, we'll fall back to the item's average rating.
3. **Global Mean**: Acts as an ultimate safety net for any user/item pair that has no user or item ratings.

In [ ]:
R_df = pd.DataFrame(R)
R_df.replace(0, np.nan, inplace=True)

global_mean = train_df['stars'].mean()

# User mean first (captures personal preference baseline)
user_means_series = R_df.mean(axis=1, skipna=True)
R_df = R_df.apply(lambda row: row.fillna(user_means_series[row.name]), axis=1)

# Item mean second (for users with no ratings on an item)
item_means = R_df.mean(skipna=True)
item_means = item_means.fillna(global_mean)
R_df = R_df.apply(lambda col: col.fillna(item_means[col.name]))

# Global mean as final safety net
R_df.fillna(global_mean, inplace=True)
R_im = R_df.values

## SVD

Here, we apply SVD to factorize our imputed matrix into three components: $U$, $\Sigma$, and $V^T$, same as what we discussed in lecture and in class examples. Through trial and error, we found by setting $k=2$ and restricting the factorization to the top 2 latent features we got the best results. What this really means is that we're reducing dimensionality and filtering out noise giving us improved predictions.

In [ ]:
k = 2
U, s, Vt = svds(R_im, k=k)
idx = np.argsort(s)[::-1]
U, s, Vt = U[:, idx], s[idx], Vt[idx, :]

predicted_ratings = U @ np.diag(s) @ Vt

## Evaluation and Metrics

With the predicted ratings calculated, we'll generate the top 30 recommendations for every user in test_df. Items that a user has already rated in the training set are masked out to ensure we are only recommending for the new items. Then, we calculate ranking metrics (Hit@K, NDCG@K) and rating error metrics (MAE, RMSE, R-squared) to evaluate.

In [70]:
test_users = test_df['user_id'].unique()
predictions = {}

for user in test_users:
    if user in user_to_idx:
        u_idx = user_to_idx[user]

        user_preds = predicted_ratings[u_idx, :].copy()

        already_rated_indices = np.where(R[u_idx, :] > 0)[0]
        user_preds[already_rated_indices] = -999.0

        top_indices = user_preds.argsort()[-30:][::-1]
        predictions[user] = [idx_to_item[i] for i in top_indices]
    else:
        predictions[user] = []

true_stars_list = []
predicted_stars_list = []

for row in test_df.itertuples():
    user = row.user_id
    item = row.business_id
    true_rating = row.stars

    # Only evaluate if both user and item were in the training set (no cold-start)
    if user in user_to_idx and item in item_to_idx:
        u_idx = user_to_idx[user]
        i_idx = item_to_idx[item]

        # Get the predicted rating
        pred_rating = predicted_ratings[u_idx, i_idx]

        true_stars_list.append(true_rating)
        predicted_stars_list.append(pred_rating)

metrics = evaluate_model(predictions, test_df)
rating_metrics = evaluate_rating_metrics(true_stars_list, predicted_stars_list)

metrics.update(rating_metrics)
results_df = pd.DataFrame([metrics]).round(4)
results_df.index = [f'Collaborative k={k}']
display(results_df)

,Hit@10,Hit@20,Hit@30,NDCG@10,NDCG@20,NDCG@30,MAE,RMSE,R2
"Collaborative (SVD no demean, k=2)",0.0719,0.1204,0.1637,0.0357,0.048,0.0572,0.9227,1.22,0.044
